In [0]:
bronze_schemapath  = "`bronze_catalog`.default"
silver_schemapath = "`silver_catalog`.default"

In [0]:
tables = spark.catalog.listTables(dbName=bronze_schemapath)

In [0]:
fivetran_ingested_column_to_remove = [
    "_file", 	
    "_fivetran_synced", 
    "_modified",
    "_line"
]

In [0]:
def remove_fivetran_ingested_column(df):
    for column in fivetran_ingested_column_to_remove:
        df = df.drop(column)
    return df

In [0]:
# Read from bronze, remove metadata columns, and write to silver
df = spark.read.table(f"{bronze_schemapath}.opportunity_table")
df_clean = remove_fivetran_ingested_column(df)
#df_clean.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schemapath}.opportunity_table")

In [0]:
from pyspark.sql.functions import to_date, col, row_number, lit, expr, regexp_replace, upper,to_timestamp
from pyspark.sql.window import Window


df = spark.read.table(f"{silver_schemapath}.opportunity_table")

# Convert date columns to proper date type
df = df.withColumn("start_date", to_date(col("start_date"), "dd-MM-yyyy"))
df = df.withColumn("end_date", to_date(col("end_date"), "dd-MM-yyyy"))

# Remove duplicates
df = df.dropDuplicates()

# Clean revenue_amount: remove nulls and ensure numeric type
df = df.withColumn(
    "revenue_amount",
    regexp_replace(col("revenue_amount"), '[$£]', '')
)
df = df.withColumn(
    "revenue_amount",
    regexp_replace(col("revenue_amount"), '[^0-9\\.-]', '')
)
df = df.withColumn(
    "revenue_amount",
    col("revenue_amount").cast("double")
)
# timestamp upto seconds created_timestamp column
df = df.withColumn("created_timestamp", to_timestamp(col("created_timestamp"), "dd-MM-yyyy HH:mm"))
# Clean close_status: standardize to uppercase
df = df.withColumn("close_status", upper(col("close_status")))

display(df)
#df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schemapath}.opportunity_table")

In [0]:
opportunity_df = spark.read.table(f"{silver_schemapath}.opportunity_table")

customer_df = spark.read.table(f"{silver_schemapath}.customer_table")
country_master_df = spark.read.table(f"{silver_schemapath}.country_table")
fx_rate_df = spark.read.table(f"{silver_schemapath}.fixrate_table")

opportunity_df = opportunity_df.alias("o") \
    .join(customer_df.alias("c"), col("o.customer_id") == col("c.customer_id"), "left") \
    .select("o.*", "c.country").alias("co") \
    .join(country_master_df.alias("cm"), col("co.country") == col("cm.country_code"), "left") \
    .select("co.*", "cm.currency_code").alias("coc") \
    .join(fx_rate_df.alias("fx"), col("coc.currency_code") == col("fx.currency_code"), "left") \
    .select("coc.*", "fx.fx_rate_to_gbp")

In [0]:
opportunity_df=opportunity_df.withColumn("revenue_in_gpb",col("revenue_amount")*col("fx_rate_to_gbp"))

In [0]:
from pyspark.sql.functions import when

opportunity_df = opportunity_df.withColumn(
    "contract_months",
    when(col("contract_term") == "Monthly", 1)
    .when(col("contract_term") == "Yearly", 12)
)
opportunity_df = opportunity_df.withColumn(
    "mrr_in_gpb",
    col("revenue_in_gpb") / col("contract_months")
)